# 📊 Retail Sales Data — Exploratory Data Analysis

Objective: uncover sales trends, customer behaviour patterns and actionable business insights.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/retail_sales_dataset.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

## 1. Initial Inspection
Check shape, data types and missing values.

In [ ]:
print('Shape:', df.shape)
display(df.dtypes.to_frame('Data Type'))
display(df.isnull().sum().to_frame('Missing Values'))


## 2. Descriptive Statistics
Mean, median, mode and standard deviation for all numerical columns.

In [ ]:
numeric = df.select_dtypes(include='number')
stats = pd.DataFrame({
    'Mean': numeric.mean(),
    'Median': numeric.median(),
    'Mode': numeric.mode().iloc[0],
    'Standard Deviation': numeric.std()
})
stats

## 3. Monthly and Quarterly Sales Trends

In [ ]:
monthly = df.set_index('Date')['Total Amount'].resample('MS').sum()
quarterly = df.set_index('Date')['Total Amount'].resample('QS').sum()

fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(marker='o', ax=ax)
ax.set_title('Monthly Sales Trend')
ax.set_ylabel('Total Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Observation:** Monthly sales fluctuate across the year. Monthly aggregation helps identify short-term peaks and weaker periods.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
quarterly.plot(marker='o', ax=ax)
ax.set_title('Quarterly Sales Trend')
ax.set_ylabel('Total Sales')
plt.tight_layout()
plt.show()


**Observation:** Quarterly aggregation smooths monthly variation and makes broader revenue cycles easier to compare.

## 4. Customer Demographics

In [ ]:
df['Age Group'] = pd.cut(df['Age'], bins=[17,25,35,45,55,65], labels=['18–25','26–35','36–45','46–55','56–64'])
fig, ax = plt.subplots(figsize=(8, 4))
df['Age Group'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Customer Age Group Distribution')
ax.set_ylabel('Customers / Transactions')
plt.tight_layout()
plt.show()


**Observation:** The age-group distribution shows which customer segments contribute the largest number of transactions.

In [ ]:
gender_sales = df.groupby('Gender')['Total Amount'].sum().sort_values(ascending=False)
gender_sales.plot(kind='bar', figsize=(6,4), title='Revenue by Gender')
plt.ylabel('Total Revenue')
plt.tight_layout()
plt.show()

**Observation:** Comparing revenue by gender helps identify whether customer value is balanced across the two gender segments.

## 5. Product Category Analysis

In [ ]:
category_sales = df.groupby('Product Category')['Total Amount'].sum().sort_values(ascending=False)
category_sales.plot(kind='bar', figsize=(8,4), title='Revenue by Product Category')
plt.ylabel('Total Revenue')
plt.tight_layout()
plt.show()


**Observation:** Category revenue highlights which product groups deserve stronger inventory and marketing attention.

### Top 10 analysis — dataset limitation
The supplied dataset has `Product Category` but no `Product Name`. Therefore a genuine top-10 product ranking cannot be calculated. We use the top 10 highest-value transactions as a transparent substitute.

In [ ]:
top10 = df.nlargest(10, 'Total Amount')[['Transaction ID','Product Category','Quantity','Total Amount']]
top10

## 6. Correlation Heatmap

In [ ]:
corr_cols = ['Age','Quantity','Price per Unit','Total Amount']
plt.figure(figsize=(8,5))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='Blues')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

**Observation:** Quantity and Total Amount are strongly related because transaction revenue depends on units and price. Price per Unit helps explain differences in order value.

## 7. Additional Non-Obvious Insight
Compare revenue contribution with transaction volume by age group.

In [ ]:
age_summary = df.groupby('Age Group', observed=False).agg(
    Transactions=('Transaction ID','count'),
    Revenue=('Total Amount','sum')
)
age_summary['Revenue per Transaction'] = age_summary['Revenue'] / age_summary['Transactions']
age_summary

**Observation:** A segment can have many transactions without producing the highest revenue per transaction. This distinction can guide targeted promotions and premium-product campaigns.

## 8. Conclusion & Actionable Recommendations

1. Prioritize high-revenue categories with inventory planning and targeted campaigns.
2. Use monthly and quarterly trends to time promotions during weaker sales periods.
3. Segment campaigns by age group and focus premium offers on segments with higher revenue per transaction.
4. Monitor quantity and unit-price combinations to identify opportunities for bundles and upselling.
